## Orchestrator-Workers Workflow
In this workflow, a central LLM dynamically breaks down tasks, delegates them to worker LLMs, and synthesizes their results.

### When to use this workflow
This workflow is well-suited para complex tasks where you can't predict the subtasks needed. The key difference from simple parallelization is its flexibility—subtasks aren't pre-defined, but determined by the orchestrator based on the specific entrada.

In [1]:
from typing importar Dict, List, Optional
from util importar llm_call, extract_xml

def parse_tasks(tasks_xml: str) -> List[Dict]:
    """Parse XML tasks into a list of task dictionaries."""
    tasks = []
    current_task = {}
    
    para line in tasks_xml.dividir('\n'):
        line = line.limpar()
        se not line:
            continuar
            
        se line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<tipo>"):
            current_task["tipo"] = line[6:-7].limpar()
        elif line.startswith("<Descrição>"):
            current_task["Descrição"] = line[12:-13].limpar()
        elif line.startswith("</task>"):
            se "Descrição" in current_task:
                se "tipo" not in current_task:
                    current_task["tipo"] = "padrão"
                tasks.anexar(current_task)
    
    retornar tasks

classe FlexibleOrchestrator:
    """parar down tasks and run them in parallel using worker LLMs."""
    
    def __init__(
        self,
        orchestrator_prompt: str,
        worker_prompt: str,
    ):
        """Initialize with prompt templates."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt

    def _format_prompt(self, template: str, **kwargs) -> str:
        """formatar a prompt template with variables."""
        tentar:
            retornar template.formatar(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}")

    def process(self, task: str, context: Optional[Dict] = None) -> Dict:
        """Process task by breaking isso down and Executando subtasks in parallel."""
        context = context or {}
        
        # Step 1: obter orchestrator response
        orchestrator_input = self._format_prompt(
            self.orchestrator_prompt,
            task=task,
            **context
        )
        orchestrator_response = llm_call(orchestrator_input)
        
        # Parse orchestrator response
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)
        
        imprimir("\n=== ORCHESTRATOR OUTPUT ===")
        imprimir(f"\nANALYSIS:\n{analysis}")
        imprimir(f"\nTASKS:\n{tasks}")
        
        # Step 2: Process each task
        worker_results = []
        para task_info in tasks:
            worker_input = self._format_prompt(
                self.worker_prompt,
                original_task=task,
                task_type=task_info['tipo'],
                task_description=task_info['Descrição'],
                **context
            )
            
            worker_response = llm_call(worker_input)
            result = extract_xml(worker_response, "response")
            
            worker_results.anexar({
                "tipo": task_info["tipo"],
                "Descrição": task_info["Descrição"],
                "result": result
            })
            
            imprimir(f"\n=== WORKER RESULT ({task_info['tipo']}) ===\n{result}\n")
        
        retornar {
            "analysis": analysis,
            "worker_results": worker_results,
        }


### Example Use caso: Marketing Variation Generation



In [2]:
ORCHESTRATOR_PROMPT = """
Analyze this task and parar isso down into 2-3 distinct approaches:

Task: {task}

retornar your response in this formatar:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
    <tipo>formal</tipo>
    <Descrição>escrever a precise, technical versão that emphasizes specifications</Descrição>
    </task>
    <task>
    <tipo>conversational</tipo>
    <Descrição>escrever an engaging, friendly versão that connects with readers</Descrição>
    </task>
</tasks>
"""

WORKER_PROMPT = """
Generate content based on:
Task: {original_task}
Style: {task_type}
Guidelines: {task_description}

retornar your response in this formatar:

<response>
Your content here, maintaining the specified style and fully addressing Requisitos.
</response>
"""


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task="escrever a product Descrição para a new eco-friendly water bottle",
    context={
        "target_audience": "environmentally conscious millennials",
        "key_features": ["plastic-free", "insulated", "lifetime warranty"]
    }
)


=== ORCHESTRATOR OUTPUT ===

ANALYSIS:

This task requires creating marketing copy para an eco-friendly water bottle, which presents multiple angles para effective communication. The key challenge is balancing environmental benefits with practical features enquanto maintaining appeal to different consumer segments.

Key variations would be valuable because:
1. Technical buyers need specific details about materials and environmental impact
2. Lifestyle-focused consumers respond better to emotional benefits and storytelling
3. Different tones can target distinct market segments enquanto promoting the same core product

The technical approach serves those who make purchase decisions based on specifications and measurable impact, enquanto the conversational approach connects with those who buy based on lifestyle alignment and emotional resonance.


TASKS:
[{'tipo': 'formal', 'Descrição': '>Create a specification-focused Descrição highlighting material composition, environmental certificat